In [29]:
import sys
sys.path.append("scripts/") 

In [47]:
import numpy as np
import os
import sys
import datetime
from torch.utils.tensorboard import SummaryWriter
from sklearn.preprocessing import StandardScaler
from nn_library import dccDataset, train, model_registry
from tqdm import tqdm 

In [48]:
import torch
import torch.utils.data as torch_data


In [49]:
ML_DATA_PATH = "/project/scratch/p200631/Silvana/gpu_utilization/results/60-random/ml.npz"

In [50]:
ml_data = np.load(ML_DATA_PATH)
X_train, y_train, X_test, y_test = ml_data['X_train'], ml_data['y_train'],ml_data['X_test'],ml_data['y_test']
n_classes = len(np.unique(y_train))

In [51]:
X_train

array([[[8.1000e+01, 3.5000e+01, 1.4090e+03, ..., 6.6000e+01,
         6.7000e+01, 2.0483e+02],
        [8.7000e+01, 4.0000e+01, 1.4090e+03, ..., 6.5000e+01,
         6.8000e+01, 6.3980e+01],
        [8.7000e+01, 4.0000e+01, 1.4090e+03, ..., 6.6000e+01,
         6.8000e+01, 1.5254e+02],
        ...,
        [8.0000e+01, 3.8000e+01, 1.4090e+03, ..., 6.5000e+01,
         7.0000e+01, 6.7300e+01],
        [8.0000e+01, 3.8000e+01, 1.4090e+03, ..., 6.6000e+01,
         7.0000e+01, 1.6653e+02],
        [8.1000e+01, 3.6000e+01, 1.4090e+03, ..., 6.5000e+01,
         6.9000e+01, 9.8600e+01]],

       [[1.0000e+02, 1.0000e+00, 2.0689e+04, ..., 4.5000e+01,
         4.1000e+01, 4.6810e+01],
        [1.0000e+02, 1.0000e+00, 2.0689e+04, ..., 4.5000e+01,
         4.3000e+01, 1.9456e+02],
        [9.5000e+01, 1.9000e+01, 2.0689e+04, ..., 4.8000e+01,
         4.5000e+01, 2.2878e+02],
        ...,
        [1.0000e+02, 1.0000e+00, 2.0689e+04, ..., 4.5000e+01,
         4.0000e+01, 4.6330e+01],
        [1.0

In [35]:
#standartize 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

In [36]:
# build datasets
dset_train = dccDataset(X_train,y_train)
dset_test = dccDataset(X_test,y_test)

In [37]:
seed = 2022
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

max_workers = 0
batch_size = 32

In [38]:
# torch dataloaders
train_dl = torch_data.DataLoader(dset_train, batch_size=batch_size, num_workers=max_workers, shuffle=True)
test_dl = torch_data.DataLoader(dset_test, batch_size=batch_size, num_workers=max_workers, shuffle=False)

In [39]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
DEVICE

device(type='cuda')

In [41]:
from nn_library import ts_class_naive

In [42]:
model = model_registry["ts_class_naive"](timesteps=540,
                                 series=7,
                                 hidden_size=32,
                                 output_size=n_classes,
                                 rnn_layers=2).double().to(DEVICE)

In [43]:
model

ts_class_naive(
  (rnn): LSTM(7, 32, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=34560, out_features=540, bias=True)
  (out): Linear(in_features=540, out_features=26, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (relu): LeakyReLU(negative_slope=0.01)
)

In [44]:
writer = SummaryWriter()


In [45]:
train_results = train(train_dl,
                      test_dl,
                      model,
                      "ts_class_naive",
                      num_epochs=10,
                      writer=writer)

Epoch:   0. Train Loss: 25.8672. Test Loss: 3.3940. Train Acc.: 54.17% test Acc.: 67.52%
Epoch 0 best model saved with accuracy: 67.52%
Epoch 2 best model saved with accuracy: 69.10%
Epoch 3 best model saved with accuracy: 72.01%
Epoch 4 best model saved with accuracy: 72.10%
Epoch:   5. Train Loss: 11.2600. Test Loss: 2.3096. Train Acc.: 70.83% test Acc.: 76.14%
Epoch 5 best model saved with accuracy: 76.14%
Epoch 7 best model saved with accuracy: 77.18%
Epoch 8 best model saved with accuracy: 80.74%


In [46]:
train_results

([25.867237017715418,
  18.09855813216896,
  15.97802747524607,
  13.776551767801083,
  13.043007300581403,
  11.25997860848577,
  10.67042173759869,
  10.803699530023128,
  8.92845337612262,
  8.920472272211137],
 [3.394008848357169,
  3.198166210163423,
  3.0238346369420004,
  2.6560315073981813,
  2.5780593587777108,
  2.3095558082894687,
  2.278712291779045,
  2.325239686748722,
  1.986546857805623,
  2.183476788956966],
 [0.6752329850324768,
  0.6721265179327873,
  0.6910477266308952,
  0.7201355549279864,
  0.7209827732279017,
  0.7613668455238634,
  0.759390002824061,
  0.7718158712228184,
  0.8073990398192601,
  0.7723806834227619])